## Statistical Analysis and comparison of geodesic and Procrustes distance

This notebook compares two shape distance metrics in regard to their descriptive and inferential properties.

### Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy
from itertools import combinations
from skbio.stats.distance import DistanceMatrix, permanova, permdisp, anosim
import scipy.stats as stats
from scipy.stats import pearsonr
from SqST_Functions import KNN_Bootstrapping, KNN_distMat, modeValue

### Data Preparation

Set the paths needed to the expected input files:

1. The index file is expected to be a data frame with the columns "Index", "Filename" and "Species". Further columns with categories are possible.
2. The out_path determines the saving location for all generated output files
3. The distance tables for both the procrustes distances and the geodesic distances between the specimens and the groups means are expected. The format should be a pairwise table with an upper and lower triangle the first column is expected to be 'Filename' holding the specimens.
4. The distance tables between the means for each group, same format as the specimen one.

In [ ]:
index = pd.read_csv('PATH') # Insert path to the index file for the dataset
out_path = 'PATH' # Insert path to the designated output folder
p_dist = pd.read_csv('PATH') # Insert path to the procrustes distance table
g_dist = pd.read_csv('PATH') # Insert path to the geodesic distance table
Pmean_dist = pd.read_csv('PATH') # Insert path to the procrustes distances between means table
Gmean_dist = pd.read_csv('PATH') # Insert path to the procrustes distances between means table

To prepare the loaded pairwise distances for statistical input, they need to be transformed into a numerical matrix.

In [ ]:
# Procrustes Distance
proc_dist = DistanceMatrix(np.ascontiguousarray(p_dist.set_index('Filename').to_numpy(), p_dist['Filename']))
procMean_dist = Pmean_dist.set_index('Name').to_numpy()

# Geodesic Distance
geod_dist = DistanceMatrix(np.ascontiguousarray(g_dist.set_index('Filename').to_numpy(), g_dist['Filename']))
geodMean_dist = Gmean_dist.set_index('Name').to_numpy()

idx = np.triu_indices_from(geodMean_dist, k=1)

Define the groups that are to be the categorical variable for statistical testing, here species. It is important that the groups are listed in the same order as the distance matrix it is used with, therefore one grouping per distance matrix.

In [ ]:
grouping_geo = index.set_index('Filename').loc[g_dist['Filename'],'Species'].to_numpy()
grouping_pro = index.set_index('Filename').loc[p_dist['Filename'],'Species'].to_numpy()

### Pearson correlation coefficient
To check if the two distances really percieve a similar shape geometrie
Needed is only the upper triangle of a pairwise distance matrix for the mean of the groups

In [ ]:
r, p = pearsonr(procMean_dist[idx], geodMean_dist[idx])
print("r =",r, "/ p =",p)

Also visualise the relation between the distances

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(procMean_dist[idx], geodMean_dist[idx], '.')
ax.set_xlabel("Mean Procrustes Distance")
ax.set_ylabel("Mean Geodesic Distance")
plt.show()

### PERMANOVA 

To test if the centroids of the groups, here species, are statistically separated from each other. 
The output will be a table showing the results for both distances

In [ ]:
input = [('Eigenshape', proc_dist, grouping_pro),('SRVF', geod_dist, grouping_geo)]

perm_results = []

for lab, dist, grouping in input:
	anov = permanova(dist, grouping, permutations = 9999)
	anov['R^2'] = 1 / (1 + (anov['sample size'] - anov['number of groups']) / ((anov['number of groups'] - 1) * anov['test statistic']))
	anov.name = lab
	
	perm_results.append(anov)
	
PERMANOVA = pd.DataFrame(perm_results)
PERMANOVA.to_csv(out_path + 'YOUR_FILENAME.csv')

print(PERMANOVA.loc['SRVF'])
print(PERMANOVA.loc['Eigenshape'])

#### Pairwise PERMANOVA
The pairwise comparison of each group will allow a more detailed analyses of the differences between the individual pairs.

In [ ]:
pw_results = []
species = pd.unique(index['Species'].sort_values())

for lab, dist, grouping in input:
	for spec1, spec2 in combinations(species, 2):
		spec_pair = np.isin(grouping, [spec1, spec2])
		pair_id = np.array(dist.ids)[spec_pair]
		pair_dist = dist.filter(pair_id)
		pair = grouping[spec_pair]
	
		anov = permanova(pair_dist, pair, permutations=9999)
		anov['R^2'] = 1 / (1 + (anov['sample size'] - anov['number of groups']) / ((anov['number of groups'] - 1) * anov['test statistic']))
		anov.name = f'{spec1} : {spec2}'
		anov['method'] = lab
	
		pw_results.append(anov)

PW_ANOVA = (pd.DataFrame(pw_results)
    .drop(columns =['method name', 'test statistic name', 'number of groups', 'number of permutations'])
    .sort_index())
PW_ANOVA.to_csv(out_path + 'YOUR_FILENAME.csv')

### ANOSIM
The ANOSIM tests next if the specimens within a group are more similar to each other than to specimens of another group. 

In [ ]:
anosim_results = []

for lab, dist, grouping in input:
	sim = anosim(dist, grouping, permutations = 9999)
	sim.name = lab
	
	anosim_results.append(sim)
	
ANOSIM = pd.DataFrame(anosim_results)
ANOSIM.to_csv(out_path + 'YOUR_FILENAME.csv')

print(ANOSIM.loc['SRVF'])
print(ANOSIM.loc['Eigenshape'])


#### Pairwise ANOSIM
As with the pairwise comparison for the PERMANOVA it will allow a more detailed analyses of the differences between the individual pairs.

In [ ]:
pw_anosim = []

for lab, dist, grouping in input:
	for spec1, spec2 in combinations(species, 2):
		spec_pair = np.isin(grouping, [spec1, spec2])
		pair_id = np.array(dist.ids)[spec_pair]
		pair_dist = dist.filter(pair_id)
		pair = grouping[spec_pair]
	
		sim = anosim(pair_dist, pair, permutations=9999)
	
		sim.name = f'{spec1} : {spec2}'
		sim['method'] = lab
	
		pw_anosim.append(anov)

PW_ANOSIM = (pd.DataFrame(pw_anosim)
    .drop(columns =['method name', 'test statistic name', 'number of groups', 'number of permutations'])
    .sort_index())
PW_ANOSIM.to_csv(out_path + 'YOUR_FILENAME.csv')

### PERMDISP
Both the PERMANOVA and the ANOSIM are vulnerable to heterogeneity. Therefore an evaluation of within-group dispersion is needed.

In [ ]:
disp_results = []

for lab, dist, grouping in input:
	disp = permdisp(dist, grouping, test= 'median', permutations = 9999)
	disp.name = lab
	disp_results.append(disp)
	
PERMDISP = pd.DataFrame(disp_results)
PERMDISP.to_csv(out_path + 'YOUR_FILENAME.csv')

print(PERMDISP.loc['SRVF'])
print(PERMDISP.loc['Eigenshape'])

### k-NN Analysis
Classification accuracy is tested using a k-nearest-neighbour bootstrap analysis

In [ ]:
classes = np.unique(list(index['Species']))

kNNGeod = KNN_Bootstrapping(
    index,
    g_dist,
    classes,
    nBootstrapping=100, # default 50
    classType=str, # default str
    trainingProportion=0.5, # set to half the dataset default 0
    kNeighbours=np.int_(np.linspace(3, 12, 10)), # default 3...12
    nameVariable='Filename', # default is Name
    classVariable='Species') 

# Calculation of the mean, standard deviation and confidence intervall for the accuracy

g_mean_f1 = np.mean(kNNGeod['allF1Scores'])
g_sd_f1 = np.std(kNNGeod['allF1Scores'], ddof=1)
g_ci = np.quantile(kNNGeod['allF1Scores'], [0.025, 0.975])

print('Top F1 Score [%]: ', kNNGeod['topF1Score']*100,'\nMean F1 Score [%]: ', g_mean_f1*100, '\nSD: ',g_sd_f1, '\n95% CI: ', g_ci)

Then run the k-NN with the second distance metric on the same trainings-set to ensure comparability of results.

In [ ]:
kNNProc = KNN_Bootstrapping(
    index,
    p_dist,
    classes,
    nBootstrapping=100, # default 50
    classType=str, # default str
    trainingProportion=0.5, # default 0
    useTrainingSample=True,
    fixedTrainingSample=kNNGeod['allTrainingSamples'],
    kNeighbours=np.int_(np.linspace(3, 12, 10)), # default 3...12
    nameVariable='Filename', # default is Name
    classVariable='Species') 

# Calculation of the mean, standard deviation and confidence intervall for the accuracy

p_mean_f1 = np.mean(kNNProc['allF1Scores'])
p_sd_f1 = np.std(kNNProc['allF1Scores'], ddof=1)
p_ci = np.quantile(kNNProc['allF1Scores'], [0.025, 0.975])

print('Top F1 Score [%]: ', kNNProc['topF1Score']*100,'\nMean F1 Score [%]: ', p_mean_f1*100, '\nSD: ',p_sd_f1, '\n95% CI: ', p_ci)

Comparison of the kNN output

In [ ]:
diff = kNNGeod['allF1Scores'] - kNNProc['allF1Scores']
diff_mean, diff_sd = diff.mean(), diff.std(ddof=1)
diff_ci = np.quantile(diff, [0.025, 0.975])
print("Mean Difference between the metrics (SRV-ES) :", diff_mean, ", standard deviation:", diff_sd, ", 95% confidence intervall:" , diff_ci)

Save the results as one CSV per key

In [ ]:
all_names = list(index['Filename'])
knns = [('SRVF', kNNGeod), ('ES', kNNProc)]

for lab, kNN in knns:
    for key, value in kNN.items():
        # Save the best run with the highest F1 score
        kNN['topClassification'].to_csv(out_path + 'YOUR_'+lab+'_FILENAME.csv',index=False)
        # Save the F1 scores for each run
        pd.DataFrame(kNN['allF1Scores'].T).to_csv(out_path + 'YOUR_'+lab+'_FILENAME.csv')
        # Save training sets for each run 
        pd.DataFrame(kNN['allTrainingSamples']).T.to_csv(out_path + 'YOUR_'+lab+'_FILENAME.csv',index=False)
        # Save all classification sets
        (pd.concat([allClass for allClass in kNN['allClassificationResults']],axis=1)
            .to_csv(out_path + 'YOUR_'+lab+'_FILENAME.csv',index=False))
        # Save training and testing samples for the run with the highest F1 score
        topTraining = kNN['topTrainingTesting']['train']
        topTesting = kNN['topTrainingTesting']['test']
        training_testing = ['train' if nm in topTraining else 'test' for nm in all_names]
        (pd.DataFrame([all_names,training_testing])
            .T
            .rename(columns={0: 'Filename', 1: 'Category'})
            .sort_values('Filename')
            .to_csv(out_path + 'YOUR_'+lab+'_FILENAME.csv', index=False)
        )